# Pandas Refresher — Addis Ababa Urban Analytics Edition

This notebook refreshes core Pandas skills using data shaped like what you'll actually work with: sub-city populations, transit stops, and housing/income figures for Addis Ababa.

**How to use this notebook:**
- Each section has a short explanation, then an **Exercise** cell where you write code.
- Right below each exercise is a **Solution** cell (collapsed content — try the exercise first before peeking).
- Run cells top to bottom with `Shift+Enter`.

Sections:
1. Series & DataFrame basics
2. Selection & filtering
3. GroupBy & aggregation
4. Merging & joining datasets
5. Handling missing data
6. Apply, lambda & sorting
7. Pivot tables
8. Mini challenge (combines everything)


## 0. Setup

Run this first — it creates all the sample datasets we'll use throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 100)

print("Pandas version:", pd.__version__)

In [ ]:
# Sample dataset 1: Sub-city population & area (Addis Ababa, illustrative figures)
subcity_data = {
    "sub_city": [
        "Bole", "Yeka", "Kirkos", "Lideta", "Arada",
        "Addis Ketema", "Akaky Kaliti", "Nifas Silk-Lafto",
        "Kolfe Keranio", "Gulele"
    ],
    "population": [
        328900, 401000, 235100, 210300, 225700,
        345600, 195400, 380200, 372100, 296800
    ],
    "area_km2": [
        122.8, 85.9, 14.6, 9.4, 9.9,
        11.4, 118.1, 68.3, 61.3, 30.2
    ],
    "avg_income_birr": [
        18500, 12200, 15800, 11900, 13400,
        9800, 8600, 14200, 10100, 12800
    ]
}
subcity_df = pd.DataFrame(subcity_data)
subcity_df

In [ ]:
# Sample dataset 2: Transit stops (points, simplified — no real geometry needed for this notebook)
np.random.seed(42)
modes = ["Anbessa Bus", "Sheger Bus", "Light Rail", "Minibus Taxi"]
subcities_list = subcity_df["sub_city"].tolist()

n_stops = 60
transit_stops = pd.DataFrame({
    "stop_id": [f"STOP-{i:03d}" for i in range(1, n_stops + 1)],
    "sub_city": np.random.choice(subcities_list, n_stops),
    "mode": np.random.choice(modes, n_stops, p=[0.35, 0.25, 0.1, 0.3]),
    "daily_boardings": np.random.randint(50, 3000, n_stops)
})

# Deliberately introduce some missing values, like real-world messy data
transit_stops.loc[[3, 15, 27, 40], "daily_boardings"] = np.nan
transit_stops.loc[[7, 22], "mode"] = None

transit_stops.head(10)

## 1. Series & DataFrame basics

A **Series** is a single column (1D). A **DataFrame** is a table (2D) — a dict of Series sharing an index.

Key attributes/methods: `.shape`, `.columns`, `.dtypes`, `.info()`, `.describe()`, `.head()`, `.tail()`

In [ ]:
# Example
print(subcity_df.shape)
print(subcity_df.dtypes)
subcity_df.describe()

### Exercise 1.1
Extract the `population` column as a Series and print its mean, max, and min.

In [1]:
# Your code here


<details>
<summary>Solution 1.1 (click to expand)</summary>

```python
pop_series = subcity_df["population"]
print("Mean:", pop_series.mean())
print("Max:", pop_series.max())
print("Min:", pop_series.min())
```
</details>

### Exercise 1.2
Add a new column `pop_density` = `population` / `area_km2`, rounded to 1 decimal place.

In [ ]:
# Your code here


<details>
<summary>Solution 1.2</summary>

```python
subcity_df["pop_density"] = (subcity_df["population"] / subcity_df["area_km2"]).round(1)
subcity_df
```
</details>

## 2. Selection & filtering

- `.loc[]` — label-based selection
- `.iloc[]` — position-based selection
- Boolean masking — `df[df["col"] > value]`

### Exercise 2.1
Select all sub-cities with `pop_density` greater than 5000 people/km². Return only the `sub_city` and `pop_density` columns.

(Run Exercise 1.2's solution first if you haven't, so `pop_density` exists.)

In [ ]:
# Your code here


<details>
<summary>Solution 2.1</summary>

```python
dense_areas = subcity_df.loc[subcity_df["pop_density"] > 5000, ["sub_city", "pop_density"]]
dense_areas
```
</details>

### Exercise 2.2
From `transit_stops`, select all rows where `mode` is `"Light Rail"` **and** `daily_boardings` is greater than 1000.

Tip: combine conditions with `&`, and wrap each condition in parentheses.

In [ ]:
# Your code here


<details>
<summary>Solution 2.2</summary>

```python
busy_rail = transit_stops[(transit_stops["mode"] == "Light Rail") & (transit_stops["daily_boardings"] > 1000)]
busy_rail
```
</details>

## 3. GroupBy & aggregation

This is probably the single most useful Pandas skill for urban analytics — e.g. "average boardings per sub-city" or "total stops per mode".

Pattern: `df.groupby("column").agg(...)`

In [ ]:
# Example: total daily boardings per mode
transit_stops.groupby("mode")["daily_boardings"].sum()

### Exercise 3.1
Group `transit_stops` by `sub_city` and find the **count** of stops and the **mean** daily boardings per sub-city. Sort by stop count, descending.

Tip: `.agg({"stop_id": "count", "daily_boardings": "mean"})`

In [ ]:
# Your code here


<details>
<summary>Solution 3.1</summary>

```python
stop_summary = (
    transit_stops.groupby("sub_city")
    .agg(stop_count=("stop_id", "count"), avg_boardings=("daily_boardings", "mean"))
    .sort_values("stop_count", ascending=False)
)
stop_summary
```
</details>

### Exercise 3.2
Which sub-city has the **lowest** average income per person relative to population density? (i.e., sort `subcity_df` by `avg_income_birr` ascending and show the top 3 rows with `sub_city`, `avg_income_birr`, and `pop_density`.)

In [ ]:
# Your code here


<details>
<summary>Solution 3.2</summary>

```python
lowest_income = subcity_df.sort_values("avg_income_birr").head(3)[["sub_city", "avg_income_birr", "pop_density"]]
lowest_income
```
</details>

## 4. Merging & joining datasets

This is essential when combining, e.g., transit stop data with sub-city demographic data — exactly like your transit accessibility project.

`pd.merge(left, right, on="key_column", how="left"/"right"/"inner"/"outer")`

### Exercise 4.1
Merge `stop_summary` (from Exercise 3.1 — recompute it here if needed) with `subcity_df` on `sub_city`, using a **left join** so all sub-cities appear even if they have zero stops.

Then create a new column `stops_per_10k_pop` = `stop_count` / (`population` / 10000).

In [ ]:
# Your code here


<details>
<summary>Solution 4.1</summary>

```python
stop_summary = (
    transit_stops.groupby("sub_city")
    .agg(stop_count=("stop_id", "count"), avg_boardings=("daily_boardings", "mean"))
    .reset_index()
)

merged = pd.merge(subcity_df, stop_summary, on="sub_city", how="left")
merged["stop_count"] = merged["stop_count"].fillna(0)
merged["stops_per_10k_pop"] = merged["stop_count"] / (merged["population"] / 10000)
merged.sort_values("stops_per_10k_pop")
```

Note: this pattern — merge, then compute an accessibility ratio — is exactly what underlies your transit accessibility project, just without the spatial buffering step.
</details>

## 5. Handling missing data

Real datasets (OSM exports especially) are full of gaps. Key tools:
- `.isna()` / `.notna()` — detect missing values
- `.dropna()` — remove rows/columns with missing values
- `.fillna()` — fill missing values with a default, mean, or forward/backward fill

In [ ]:
# Check how much is missing
transit_stops.isna().sum()

### Exercise 5.1
Fill missing `daily_boardings` values with the **mean** boardings for that stop's `mode` (not the overall mean — group-wise fill is more accurate).

Tip: `df["col"] = df.groupby("group_col")["col"].transform(lambda x: x.fillna(x.mean()))`

In [ ]:
# Your code here


<details>
<summary>Solution 5.1</summary>

```python
transit_stops["daily_boardings"] = (
    transit_stops.groupby("mode")["daily_boardings"]
    .transform(lambda x: x.fillna(x.mean()))
)
transit_stops.isna().sum()
```

**Subtlety worth knowing:** by default, `groupby()` drops rows where the group key itself (`mode` here) is missing — so rows with `mode = None` get `NaN` back from `.transform()`, even if their `daily_boardings` was originally fine. That's harmless here since Exercise 5.2 drops those rows anyway, but it's a common gotcha worth remembering.
</details>

### Exercise 5.2
Drop any remaining rows where `mode` is missing (since we can't reasonably guess a categorical value).

In [ ]:
# Your code here


<details>
<summary>Solution 5.2</summary>

```python
transit_stops = transit_stops.dropna(subset=["mode"])
transit_stops.isna().sum()
```
</details>

## 6. Apply, lambda & sorting

`.apply()` lets you run a custom function across rows or columns. Combined with `lambda`, it's great for quick transformations that don't have a built-in method.

### Exercise 6.1
Create a new column `income_tier` in `subcity_df` that labels each sub-city as `"Low"` if `avg_income_birr < 11000`, `"Mid"` if between 11000 and 15000, and `"High"` otherwise.

Tip: write a small function and use `.apply()`, or use `pd.cut()` for a cleaner approach.

In [ ]:
# Your code here


<details>
<summary>Solution 6.1</summary>

```python
def income_tier(income):
    if income < 11000:
        return "Low"
    elif income <= 15000:
        return "Mid"
    else:
        return "High"

subcity_df["income_tier"] = subcity_df["avg_income_birr"].apply(income_tier)
subcity_df[["sub_city", "avg_income_birr", "income_tier"]]
```

Cleaner alternative with `pd.cut`:
```python
subcity_df["income_tier"] = pd.cut(
    subcity_df["avg_income_birr"],
    bins=[0, 11000, 15000, float("inf")],
    labels=["Low", "Mid", "High"]
)
```
</details>

## 7. Pivot tables

`pd.pivot_table()` reshapes data — e.g., "boardings by sub-city, broken down by mode" as a matrix. Very useful for quick cross-tabulations before mapping.

### Exercise 7.1
Create a pivot table showing **total daily_boardings**, with `sub_city` as rows and `mode` as columns. Fill missing combinations with 0.

In [ ]:
# Your code here


<details>
<summary>Solution 7.1</summary>

```python
pivot = pd.pivot_table(
    transit_stops,
    values="daily_boardings",
    index="sub_city",
    columns="mode",
    aggfunc="sum",
    fill_value=0
)
pivot
```
</details>

## 8. Mini challenge — putting it all together

This mirrors the kind of workflow you'd run before even opening QGIS: clean → merge → aggregate → rank.

**Task:**
1. Start from the raw `transit_stops` and `subcity_df` datasets.
2. Clean missing data (fill boardings by mode mean, drop missing modes).
3. Aggregate: total boardings and stop count per sub-city.
4. Merge with `subcity_df`.
5. Compute a simple **"transit access score"** = `stop_count / (population / 10000)` (stops per 10k residents).
6. Return the 3 sub-cities with the **lowest** transit access score — these would be your "underserved areas" candidates.

Try building this as one clean chain of code below.

In [ ]:
# Your code here — the full mini-pipeline


<details>
<summary>Solution 8 (full pipeline)</summary>

```python
# 1-2: Clean
clean_stops = transit_stops.copy()
clean_stops["daily_boardings"] = (
    clean_stops.groupby("mode")["daily_boardings"].transform(lambda x: x.fillna(x.mean()))
)
clean_stops = clean_stops.dropna(subset=["mode"])

# 3: Aggregate
agg = clean_stops.groupby("sub_city").agg(
    stop_count=("stop_id", "count"),
    total_boardings=("daily_boardings", "sum")
).reset_index()

# 4: Merge
merged = pd.merge(subcity_df, agg, on="sub_city", how="left")
merged["stop_count"] = merged["stop_count"].fillna(0)
merged["total_boardings"] = merged["total_boardings"].fillna(0)

# 5: Access score
merged["transit_access_score"] = merged["stop_count"] / (merged["population"] / 10000)

# 6: Lowest 3 — underserved candidates
underserved = merged.sort_values("transit_access_score").head(3)[
    ["sub_city", "population", "stop_count", "transit_access_score"]
]
underserved
```

This is the same core logic your `addis-transit-access` project uses, minus the spatial buffering — good bridge between the Pandas refresher and your GeoPandas work.
</details>

## Next steps

- If everything above felt comfortable, you're ready to move back into GeoPandas work (e.g., WorldPop raster integration for your transit project).
- If groupby/merge felt shaky, it's worth redoing sections 3-4 with a different grouping column (e.g., group by `income_tier` instead of `sub_city`) for extra practice.
